In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q langchain langchain-community langchain-text-splitters chromadb sentence-transformers groq unstructured langchain-groq langchain-chroma

In [3]:
!pip install -U langchain langchain-core langchain-community langchain-text-splitters langchain-chroma langchain-groq chromadb sentence-transformers unstructured

In [21]:
import os

from langchain_community.document_loaders import TextLoader,PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA

In [ ]:
os.environ['GROQ_API_KEY']="your api"

In [18]:
import requests
url="https://www.cse.iitd.ac.in/~pkalra/csl101/Python-OOP.pdf"
response=requests.get(url)

In [19]:
with open("python_inbuildfunction.pdf","wb")as f:
  f.write(response.content)

In [23]:
loader = PyPDFLoader("python_inbuildfunction.pdf")
documents = loader.load()

print("Pages loaded:", len(documents))

Pages loaded: 19


In [24]:
#chunking
text_splitter=CharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400
)

In [25]:
texts=text_splitter.split_documents(documents)

In [26]:
type(texts)

list

In [27]:
len(texts)

19

In [28]:
texts[4]

Document(metadata={'producer': 'Mac OS X 10.6.8 Quartz PDFContext', 'creator': 'Microsoft PowerPoint', 'creationdate': "D:20120419045234Z00'00'", 'author': 'PKALRA', 'moddate': "D:20120419045234Z00'00'", 'keywords': '', 'aapl:keywords': '[]', 'source': 'python_inbuildfunction.pdf', 'total_pages': 19, 'page': 4, 'page_label': '5'}, page_content='CreaDng\t\r \xa0Classes\t\r \xa0\n5\n•Defining a class in Python is done using the class \nkeyword, followed by an indented block with the class \ncontents.\nclass <Classname>:\ndata1 = value1\n...\ndataM = valueM\ndef <function1>(self,arg1,...,argK):\n<block>\n...\ndef <functionN>(self,arg1,...,argK):\n<block>\n General Format\nClass data\nattributes\nClass \nmember \nfunctions\n•Note: the entire class code block (i.e. any attribute and \nfunction definitions) is indented.\nDeﬁning\t\r \xa0a\t\r \xa0class\t\r \xa0in\t\r \xa0Python\t\r \xa0is\t\r \xa0done\t\r \xa0using\t\r \xa0the\t\r \xa0class\t\r \xa0\nkeyword,\t\r \xa0followed\t\r \xa0by\t\r 

In [29]:
# embeddinf step 4
embeddings=HuggingFaceEmbeddings()

/tmp/ipykernel_7878/65214032.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings()
/tmp/ipykernel_7878/65214032.py:2: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings=HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), s

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [30]:
persist_directory='vector_db'

In [44]:
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=persist_directory
)

# retriever
retriever = vectordb.as_retriever()

# llm from groq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)
qa_chain = RetrievalQA.from_chain_type( llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=True )
# invoke the qa chain and get a response for user query
query = "what are the function from this pdf "
response = qa_chain.invoke({"query": query})

In [45]:
print(response["result"])

The functions defined in the class "Maths" are:

1. `subtract(self, i, j)`: This function takes two arguments, `i` and `j`, and returns their difference.
2. `add(self, x, y)`: This function takes two arguments, `x` and `y`, and returns their sum.

Both functions take an additional argument `self`, which refers to the object itself, as is typical in Python class definitions.
